# ⚡ AI Flashcard Generator + Chatbot 🗨


In [9]:
!pip install -q "gradio==3.50.2" groq PyMuPDF
import importlib, gradio
importlib.reload(gradio)
print("Gradio version:", gradio.__version__)

Gradio version: 3.50.2


## Imports

In [10]:
import gradio as gr
from groq import Groq
import fitz
import json
import re
from google.colab import userdata

## Configure API Key

In [11]:
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

MODELS = {
    "Llama 3.1 8B  (Fast)"   : "llama-3.1-8b-instant",
    "Llama 3.3 70B (Smart)"  : "llama-3.3-70b-versatile",
    "Llama 4 Scout (Latest)" : "meta-llama/llama-4-scout-17b-16e-instruct",
}

print("Groq ready!")

Groq ready!


## Chat Logic

In [12]:
def chat_with_groq(message, history, model_name):
    model_id = MODELS.get(model_name, "llama-3.1-8b-instant")
    messages = [{"role": "system", "content": "You are a helpful AI assistant. Be concise and clear."}]
    for human, assistant in history:
        messages.append({"role": "user",      "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": message})
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            temperature=0.7,
            max_tokens=2048,
        )
        return response.choices[0].message.content
    except Exception as e:
        return "Error: " + str(e)

def respond(message, history, model_name):
    if not message.strip():
        return history, ""
    reply = chat_with_groq(message, history, model_name)
    history = history + [(message, reply)]
    return history, ""